import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")






df = pd.read_csv('student_profiles_clean.csv')

print(df.head())

df.head()

print(df.shape)

df.info()

df.isnull().sum()

df["ml_signal"] = df["python_score"] * df["machine_learning_score"]
df["backend_signal"] = df["sql_score"] * df["database_score"]
df["ai_signal"] = df["deep_learning_score"] * df["statistics_score"]
df["frontend_signal"] = df["html_score"] * df["css_score"]

career_df = df[df["career_goal"] != "Unknown"].copy()



career_features = [

    "python_score",
    "cpp_score",
    "java_score",
    "javascript_score",

    "html_score",
    "css_score",

    "sql_score",
    "database_score",

    "statistics_score",
    "probability_score",
    "linear_algebra_score",
    "calculus_score",

    "machine_learning_score",
    "deep_learning_score",

    "linux_score",
    "docker_score",
    "cloud_score",

    "interest_ai",
    "interest_data_science",
    "interest_web_dev",
    "interest_backend",
    "interest_frontend",
    "interest_mobile",
    "interest_cloud",
    "interest_cybersecurity",

    "projects_completed",
    "quiz_avg_score",
    "consistency_score",


]



career_features = career_features + [
    "ml_signal",
    "backend_signal",
    "ai_signal",
    "frontend_signal"
]

X = career_df[career_features]

y = career_df["career_goal"]

career_encoder = LabelEncoder()

y = career_encoder.fit_transform(y)
scaler = StandardScaler()



from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()


from xgboost import XGBClassifier
import numpy as np
from sklearn.metrics import accuracy_score

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,              # slightly lower helps generalization
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.1,
    objective="multi:softmax",
    num_class=len(np.unique(y_train)),
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train, y_train)

preds = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, preds))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.metrics import top_k_accuracy_score

probs = xgb.predict_proba(X_test)

top3_acc = top_k_accuracy_score(
    y_test,
    probs,
    k=3
)

print("Top-3 Accuracy:", top3_acc)

cm = confusion_matrix(y_test, xgb_preds)
cm

import numpy as np

recall = np.diag(cm) / cm.sum(axis=1)

for cls, r in zip(career_encoder.classes_, recall):
    print(f"{cls:25s} {r:.3f}")



In [2]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [3]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using Device:", device)

Using Device: mps


In [4]:
df = pd.read_csv("../data/student_profiles_clean.csv")

df.head()

,student_id,age,education_level,career_goal,experience_level,study_hours_per_week,learning_style,interest_ai,interest_data_science,interest_web_dev,...,projects_completed,quiz_attempts,quiz_avg_score,consistency_score,streak_days,target_completion_months,available_hours_weekly,recommended_next_topic,recommended_difficulty,learning_path_stage
0,STU00001,17,High School,Full Stack Developer,Beginner,76.31,Visual,46.05,75.81,55.28,...,2,11,35.25,82.32,9.0,18,13.9,SQL,Easy,2
1,STU00002,20,Bachelor,ML Engineer,Beginner,40.00,Reading/Writing,100.00,81.66,44.58,...,1,6,70.52,78.19,7.0,18,6.8,Deep Learning,Easy,2
2,STU00003,24,PhD,AI Engineer,Intermediate,6.04,Kinesthetic,63.01,62.06,61.07,...,6,8,35.40,70.38,10.0,9,8.2,Git,Medium,2
3,STU00004,29,High School,Backend Developer,Intermediate,16.51,Kinesthetic,62.79,51.71,49.12,...,5,5,27.63,38.29,0.0,9,12.0,SQL,Easy,3
4,STU00005,18,Bachelor,Software Engineer,Beginner,3.93,Auditory,53.82,56.61,30.49,...,2,3,46.08,61.56,7.0,6,10.0,Python,Easy,1


In [5]:
print(df.shape)

df.info()

df.isnull().sum()

(15000, 45)
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                15000 non-null  str    
 1   age                       15000 non-null  int64  
 2   education_level           15000 non-null  str    
 3   career_goal               15000 non-null  str    
 4   experience_level          15000 non-null  str    
 5   study_hours_per_week      15000 non-null  float64
 6   learning_style            15000 non-null  str    
 7   interest_ai               15000 non-null  float64
 8   interest_data_science     15000 non-null  float64
 9   interest_web_dev          15000 non-null  float64
 10  interest_backend          15000 non-null  float64
 11  interest_frontend         15000 non-null  float64
 12  interest_mobile           15000 non-null  float64
 13  interest_cloud            15000 non-null  float64
 14  inter

student_id                  0
age                         0
education_level             0
career_goal                 0
experience_level            0
study_hours_per_week        0
learning_style              0
interest_ai                 0
interest_data_science       0
interest_web_dev            0
interest_backend            0
interest_frontend           0
interest_mobile             0
interest_cloud              0
interest_cybersecurity      0
python_score                0
cpp_score                   0
java_score                  0
javascript_score            0
html_score                  0
css_score                   0
sql_score                   0
database_score              0
statistics_score            0
probability_score           0
linear_algebra_score        0
calculus_score              0
machine_learning_score      0
deep_learning_score         0
git_score                   0
linux_score                 0
docker_score                0
cloud_score                 0
sessions_c

In [6]:
df["math_strength"] = (
    df["statistics_score"] +
    df["probability_score"] +
    df["linear_algebra_score"] +
    df["calculus_score"]
) / 4

df["programming_strength"] = (
    df["python_score"] +
    df["cpp_score"] +
    df["java_score"] +
    df["javascript_score"]
) / 4

df["ai_strength"] = (
    df["machine_learning_score"] +
    df["deep_learning_score"]
) / 2

df["cloud_strength"] = (
    df["linux_score"] +
    df["docker_score"] +
    df["cloud_score"]
) / 3

df["engagement_score"] = (
    df["sessions_completed"] * 0.3 +
    df["projects_completed"] * 0.3 +
    df["consistency_score"] * 0.4
)

In [7]:
df["career_goal"].value_counts()

career_goal
Software Engineer         3673
Data Scientist            2156
Frontend Developer        1765
ML Engineer               1760
Backend Developer         1487
Full Stack Developer      1434
AI Engineer               1138
Cloud Engineer             569
Cybersecurity Engineer     568
Unknown                    450
Name: count, dtype: int64

In [8]:
career_df = df[
    df["career_goal"] != "Unknown"
].copy()

career_df.shape

(14550, 50)

In [9]:
career_features = [

    "python_score",
    "cpp_score",
    "java_score",
    "javascript_score",

    "html_score",
    "css_score",

    "sql_score",
    "database_score",

    "statistics_score",
    "probability_score",
    "linear_algebra_score",
    "calculus_score",

    "machine_learning_score",
    "deep_learning_score",

    "linux_score",
    "docker_score",
    "cloud_score",

    "interest_ai",
    "interest_data_science",
    "interest_web_dev",
    "interest_backend",
    "interest_frontend",
    "interest_mobile",
    "interest_cloud",
    "interest_cybersecurity",

    "projects_completed",
    "quiz_avg_score",
    "consistency_score",

    
]

In [10]:
X = career_df[career_features]

y = career_df["career_goal"]

In [11]:
career_encoder = LabelEncoder()

y = career_encoder.fit_transform(y)
scaler = StandardScaler()



In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
print(X_train.shape)
print(X_test.shape)

print(np.unique(y_train))


(11640, 28)
(2910, 28)
[0 1 2 3 4 5 6 7 8]


In [ ]:
class CareerDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx]
        )

: 

In [ ]:
train_dataset = CareerDataset(
    X_train,
    y_train
)

test_dataset = CareerDataset(
    X_test,
    y_test
)

In [ ]:

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256
)

In [ ]:
class CareerPredictor(nn.Module):

    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.net = nn.Sequential(
    nn.Linear(input_dim, 128),
    nn.ReLU(),
    nn.Dropout(0.4),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(64, num_classes)
)

    def forward(self, x):
        return self.net(x)

In [ ]:
input_dim = X_train.shape[1]

num_classes = len(
    career_encoder.classes_
)

model = CareerPredictor(
    input_dim,
    num_classes
)
model = model.to(device)

In [ ]:
# from sklearn.utils.class_weight import compute_class_weight
# weights = compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(y_train),
#     y=y_train
# )

# weights = torch.tensor(
#     weights,
#     dtype=torch.float32
# ).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
import torch.optim as optim

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()
        

        total_loss += loss.item()

        preds = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            preds == y_batch
        ).sum().item()

        total += y_batch.size(0)
    scheduler.step()

    avg_loss = total_loss / len(train_loader)

    accuracy = correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Train Acc: {accuracy:.4f}"
    )

Epoch 1/20 | Loss: 1.7912 | Train Acc: 0.4218
Epoch 2/20 | Loss: 1.2660 | Train Acc: 0.5811
Epoch 3/20 | Loss: 1.1840 | Train Acc: 0.6172
Epoch 4/20 | Loss: 1.1613 | Train Acc: 0.6221
Epoch 5/20 | Loss: 1.1427 | Train Acc: 0.6312
Epoch 6/20 | Loss: 1.1343 | Train Acc: 0.6311
Epoch 7/20 | Loss: 1.1278 | Train Acc: 0.6335
Epoch 8/20 | Loss: 1.1192 | Train Acc: 0.6347
Epoch 9/20 | Loss: 1.1172 | Train Acc: 0.6369
Epoch 10/20 | Loss: 1.1126 | Train Acc: 0.6421
Epoch 11/20 | Loss: 1.1050 | Train Acc: 0.6484
Epoch 12/20 | Loss: 1.1040 | Train Acc: 0.6425
Epoch 13/20 | Loss: 1.1020 | Train Acc: 0.6496
Epoch 14/20 | Loss: 1.0990 | Train Acc: 0.6468
Epoch 15/20 | Loss: 1.0994 | Train Acc: 0.6445
Epoch 16/20 | Loss: 1.0995 | Train Acc: 0.6448
Epoch 17/20 | Loss: 1.0967 | Train Acc: 0.6504
Epoch 18/20 | Loss: 1.0913 | Train Acc: 0.6508
Epoch 19/20 | Loss: 1.0901 | Train Acc: 0.6458
Epoch 20/20 | Loss: 1.0901 | Train Acc: 0.6507


In [ ]:
model.eval()

preds = []
actuals = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        logits = model(X_batch)

        pred = torch.argmax(
            logits,
            dim=1
        )

        preds.extend(
            pred.cpu().numpy()
        )

        actuals.extend(
            y_batch.numpy()
        )

In [ ]:
print(
    "Test Accuracy:",
    accuracy_score(
        actuals,
        preds
    )
)

Test Accuracy: 0.6584192439862543


In [ ]:
from xgboost import XGBClassifier
import numpy as np
from sklearn.metrics import accuracy_score

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,              # slightly lower helps generalization
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.1,
    objective="multi:softmax",
    num_class=len(np.unique(y_train)),
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train, y_train)

preds = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, preds))

NameError: name 'y_train' is not defined

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

preds = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, preds))

RF Accuracy: 0.6237113402061856


: 

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(np.unique(y_train)),
    eval_metric="mlogloss"
)

xgb.fit(X_train, y_train)

preds = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, preds))